In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils import dados, pivot, salvar

import numpy as np
import pandas as pd
import sklearn

In [2]:
# Dados

dados_brutos, metadados, variaveis = dados(r"data\input\dados_brutos.xlsx")

dados_limpos, metadados, variaveis = dados(r"data\input\dados_limpos.xlsx")

imput_mean_limpos, metadados, variaveis = dados(r"data\output\imput_mean_limpos.xlsx")

imput_median_limpos, metadados, variaveis = dados(r"data\output\imput_median_limpos.xlsx")

imput_knn_limpos, metadados, variaveis = dados(r"data\output\imput_knn_limpos.xlsx")

imput_mean_zscore, metadados, variaveis = dados(r"data\input\imput_mean_zscore.xlsx")

imput_median_zscore, metadados, variaveis = dados(r"data\input\imput_median_zscore.xlsx")

imput_knn_zscore, metadados, variaveis = dados(r"data\input\imput_knn_zscore.xlsx")

imput_mean_iqr, metadados, variaveis = dados(r"data\input\imput_mean_iqr.xlsx")

imput_median_iqr, metadados, variaveis = dados(r"data\input\imput_median_iqr.xlsx")

imput_knn_iqr, metadados, variaveis = dados(r"data\input\imput_knn_iqr.xlsx")


# Preparação para cálculos das métricas

In [3]:
def preparar_para_metricas(df_verdadeiro, df_predito, metadados):
    """
    Prepara dois DataFrames para cálculo de métricas.
    
    Remove metadados e garante que ambos tenham as mesmas colunas numéricas.
    
    Parameters
    ----------
    df_verdadeiro : pd.DataFrame
        DataFrame com valores verdadeiros.
    df_predito : pd.DataFrame
        DataFrame com valores preditos/imputados.
    metadados : list
        Lista de colunas de metadados a ignorar.
    
    Returns
    -------
    df_verdadeiro_prep : pd.DataFrame
        DataFrame preparado (apenas colunas numéricas comuns, sem metadados).
    df_predito_prep : pd.DataFrame
        DataFrame preparado (apenas colunas numéricas comuns, sem metadados).
    colunas_usadas : list
        Lista das colunas que serão usadas nas métricas.
    """
    # Eu presumo que os dados_verdadeiros serão uma das bases de dados produzidas até
    # o momento, e que os dados_preditos terão alguma coisa a ver com os modelos de ML.

    # Remove linhas com NaN
    df_verdadeiro = df_verdadeiro.dropna().copy()
    
    # Encontra colunas comuns (intersecção)
    colunas_comuns = set(df_verdadeiro.columns) & set(df_predito.columns)
    
    # Remove metadados. Importante, pois os cálculos de métricas só funcionam em colunas numéricas.
    # Exemplo: Nas colunas de datas (dtype=datetime), os métodos sklearn retornam TypeError
    colunas_usadas = [col for col in colunas_comuns if col not in metadados]
    
    df_verdadeiro_prep = df_verdadeiro[colunas_usadas]
    df_predito_prep = df_predito[colunas_usadas]
    
    # Usa apenas as colunas que existem em ambos 
    colunas_usadas = list(df_verdadeiro_prep.columns)
    df_predito_prep = df_predito_prep[colunas_usadas]
    
    print(f"Colunas usadas para métricas: {colunas_usadas}")
    print(f"Total de colunas: {len(colunas_usadas)}")
    
    return df_verdadeiro_prep, df_predito_prep, colunas_usadas

# Introdução artificial de NaN

In [4]:
def simular_dados_faltantes(df, pct_remover=0.05, seed=42):
    """
    Versão simplificada: cria dataset com NaN simulado.
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset base (será removido NaN primeiro).
    pct_remover : float
        Percentual de valores a remover (ex: 0.05 para 5%).
    seed : int
        Seed para reprodutibilidade.
    
    Returns
    -------
    df_sim_a : pd.DataFrame
        Dataset com NaN simulado. Cenário "a".
        Cenário "a": NaN introduzidos apenas em colunas que tinham NaN originalmente.
    mascara_a : pd.DataFrame
        Máscara marcando as coordenadas de dados faltantes do cenário "a".
    df_sim_b : pd.DataFrame
        Dataset com NaN simulado. Cenário "b".
        Cenário "b": NaN introduzidos em todas as colunas.
    mascara_b : pd.DataFrame
        Máscara marcando as coordenadas de dados faltantes do cenário "b".
    """
    
    np.random.seed(seed)
    
    from utils import separar_colunas 
    metadados, variaveis = separar_colunas(df)
    
    # Remove todas as linhas com NaN
    df_ref = df.dropna().copy()
    
    # Identifica colunas que originalmente tinham NaN
    cols_com_nan = df.columns[df.isna().any()].tolist()
    cols_com_nan = [col for col in cols_com_nan if col not in metadados]
    
    # Todas as colunas numéricas (exceto metadados)
    todas_cols = [col for col in df_ref.columns if col not in metadados]
    
    # Cria dataset simulado
    df_sim_a = df_ref.copy()
    df_sim_b = df_ref.copy()

    n_remover = int(len(df_sim_a) * pct_remover)
    
    # Estabelece os cenários
    colunas_alvo_a = cols_com_nan
    mascara_a = pd.DataFrame(False, index=df_ref.index, columns=df_ref.columns)

    colunas_alvo_b = todas_cols
    mascara_b = pd.DataFrame(False, index=df_ref.index, columns=df_ref.columns)
    
    # Remove valores aleatoriamente apenas das colunas que tinham NaN original (cenário A)
    for col in colunas_alvo_a:
        indices = np.random.choice(df_sim_a.index, n_remover, replace=False)
        df_sim_a.loc[indices, col] = np.nan
        mascara_a.loc[indices, col] = True

    # Remove valores aleatoriamente de TODAS as colunas (exceto metadados) (cenário B)
    for col in colunas_alvo_b:
        indices = np.random.choice(df_sim_b.index, n_remover, replace=False)
        df_sim_b.loc[indices, col] = np.nan
        mascara_b.loc[indices, col] = True
    
    print(f"Cenário A: {len(colunas_alvo_a)} colunas, {pct_remover*100}% removido")
    print(f"Cenário B: {len(colunas_alvo_b)} colunas, {pct_remover*100}% removido")

    return df_sim_a, mascara_a, df_sim_b, mascara_b

In [5]:
def avaliar_imputacao(df_verdadeiro, df_imputado, mascara,
                        metodo, cenario):
    """
    Interface simples para avaliar qualidade de imputação.
    
    Compara dataset verdadeiro (referência) com dataset imputado.
    
    Parameters
    ----------
    df_verdadeiro : pd.DataFrame
        Dataset verdadeiro (sem manipulações).
    df_imputado : pd.DataFrame
        Dataset que recebeu alguma transformação (imputação, etc).
    mascara : pd.DataFrame
        Máscara marcando as coordenadas de dados faltantes.
    metodo : str
        Nome do método de imputação sendo avaliado ("média", "mediana" ou "knn").
    cenario : str
        Nome do cenário sendo avaliado:
        "a" = avalia apenas colunas que tinham NaN original;
        "b" = avalia TODAS as colunas.
    
    
    Returns
    -------
    pd.DataFrame(linhas) : pd.DataFrame
        DataFrame com métricas das variáveis.
    """
    
    from utils import separar_colunas
    metadados, variaveis = separar_colunas(df_verdadeiro)
    
    # Preparar dados
    df_verd_prep, df_imput_prep, colunas = preparar_para_metricas(
        df_verdadeiro, df_imputado, metadados
    )
    
    # Alinhar máscara ao mesmo índice e colunas usadas na métrica
    mascara_prep = mascara.loc[df_verd_prep.index, colunas]

    # Cria uma lista que será convertida em DataFrame, com todas as métricas de todas as variáveis

    linhas = []

    from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

    for col in colunas:
        m = mascara_prep[col]
        if not m.any():
            continue
        y_true = df_verd_prep.loc[m, col]
        y_pred = df_imput_prep.loc[m, col]

        # Passa um dicionário para a lista "linhas", onde cada métrica de cada variável será armazenada numa linha diferente
        linhas.append({
            "variavel": col,
            "rmse": root_mean_squared_error(y_true, y_pred),
            "mae": mean_absolute_error(y_true, y_pred),
            "r2": r2_score(y_true, y_pred),
            "n_celulas": len(y_true),
            "metodo": metodo,
            "cenario": cenario,
        })
    
    # Converte a lista "linhas" para DataFrame
    return pd.DataFrame(linhas)

Fluxograma simplificado:
0) Carregar dados

1) Usar a função simular_dados_faltantes: É preciso especificar o dataframe verdadeiro e a porcentagem de dados a serem removidos (default 5%, valor utilizado para imputação via média e mediana). (i) Ela remove linhas com NaN, (ii) introduz NaN artificiais, (iii) retorna um dataframe com NaN artificiais apenas nas colunas que originalmente possuíam NaN (cenário "a"), (iv) retorna a máscara com as coordenadas desses NaN, (v) retorna um dataframe com NaN artificiais em todas as colunas, e (vi) a máscara com as coordenadas desses NaN.

2) Fazer a imputação no dataframe artificial gerado na etapa anterior. É preciso especificar qual DataFrame, e se deseja-se que o DataFrame pós imputação apresente apenas as colunas que foram manipuladas (return_reduced=True)

3) Fazer avaliação de imputação, utilizando a função avaliar_imputacao: É preciso especificar: (i) o dataframe real, (ii) o dataframe imputado, (iii) a máscara do dataframe imputado, (iv) o nome do método de imputação, e (v) qual cenário está sendo tratado ("a" ou "b").

In [6]:
# Teste
# 1
dados_limpos_a, mascara_a, dados_limpos_b, mascara_b = simular_dados_faltantes(dados_limpos)

# 2
from imputacao import mean_imput
media_limpos_a = mean_imput(dados_limpos_a, return_reduced=True)
media_limbos_b = mean_imput(dados_limpos_b, return_reduced=True)

# 3
avaliacao_media_a = avaliar_imputacao(dados_limpos, media_limpos_a, mascara=mascara_a, metodo='media', cenario='a')
avaliacao_media_b = avaliar_imputacao(dados_limpos, media_limbos_b, mascara=mascara_b, metodo='media', cenario='b')

Cenário A: 16 colunas, 5.0% removido
Cenário B: 16 colunas, 5.0% removido


TypeError: median_imput() missing 2 required positional arguments: 'metadados' and 'variaveis'

# Arquivos

In [ ]:
def avaliacao(df, modelo, metrica):

    # Listas de métricas e modelos disponíveis
    lista_modelos = ['media', 'mediana', 'knn']
    lista_metricas = ['RMSE', 'MAE', 'bias', 'r2']

    # Busca a presença de NaN no DataFrame
    if df.columns[df.isna().any()].array.size != 0:
    # Remove linhas com NaN, caso elas existam
        df = df.dropna().copy()
    # Mantém df, caso ele já não apresente NaN
    else:
        df = df.copy()   

    # Armazena as listas de metadados e variáveis de df
    from utils import separar_colunas
    metadados, variaveis = separar_colunas(df)

    # Verifica se a métrica e modelo especificados na função estão dentro das listas
    if metrica not in lista_metricas:
        raise ValueError("metrica deve conter uma das seguinte 4 métricas: 'RMSE', 'MAE', 'bias', 'r2'")
    elif modelo not in lista_modelos:
        raise ValueError("modelo deve conter um dos seguinte 3 modelos: 'media', 'mediana', 'KNN'")
    else:
        pass

    if modelo == 'media':
        from imputacao import mean_imput
        df_artificial_parcial, df_artificial_total = criar_validacao_cruzada(df, pct_remover=0.05, metadados=metadados)
        df_artificial_parcial = mean_imput(df_artificial_parcial, metadados=metadados, variaveis=variaveis, return_reduced=True)
        df_artificial_total = mean_imput(df_artificial_total, metadados=metadados, variaveis=variaveis, return_reduced=True)

    elif modelo == 'mediana':
        from imputacao import median_imput
        df_artificial_parcial, df_artificial_total = criar_validacao_cruzada(df, pct_remover=0.05, metadados=metadados)
        df_artificial_parcial = median_imput(df_artificial_parcial, metadados=metadados, variaveis=variaveis, return_reduced=True)
        df_artificial_total = median_imput(df_artificial_total, metadados=metadados, variaveis=variaveis, return_reduced=True)

    elif modelo.lower() == 'knn':
        from imputacao import knn_imput
        df_artificial_parcial, df_artificial_total = criar_validacao_cruzada(df, pct_remover=0.15, metadados=metadados)
        df_artificial_parcial = knn_imput(df_artificial_parcial, metadados=metadados, variaveis=variaveis, return_reduced=True)
        df_artificial_total = knn_imput(df_artificial_total, metadados=metadados, variaveis=variaveis, return_reduced=True)

    if metrica.lower() == 'rmse':
        from sklearn.metrics import root_mean_squared_error

        df_verd, df_art, colunas =  preparar_para_metricas(df, df_artificial_parcial, metadados)

        return root_mean_squared_error(df_verd, df_art)
    
    elif metrica.lower() == 'mae':
        from sklearn.metrics import mean_absolute_error

        df_verd, df_art, colunas =  preparar_para_metricas(df, df_artificial_parcial, metadados)

        return mean_absolute_error(df_verd, df_art)
    
    elif metrica == 'r2':
        from sklearn.metrics import r2_score

        df_verd, df_art, colunas =  preparar_para_metricas(df, df_artificial_parcial, metadados)

        return r2_score(df_verd, df_art)

In [ ]:
# Teste
df_nan_5, dados_sim_a, dados_sim_b, cols_nan = criar_validacao_cruzada(
    dados_limpos, 
    pct_remover=0.05,  # Remove 5% --> Imputação via média e mediana
    metadados=metadados
)

df_nan_15, dados_sim_a, dados_sim_b, cols_nan = criar_validacao_cruzada(
    dados_limpos, 
    pct_remover=0.15,  # Remove 15% --> Imputação via KNN
    metadados=metadados
)

from imputacao import mean_imput, median_imput, knn_imput

# Apenas colunas com NaN originalmente
imput_mean_a = mean_imput(dados_sim_a, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_median_a = median_imput(dados_sim_a, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_knn_a = knn_imput(dados_sim_a, metadados=metadados, variaveis=variaveis, return_reduced=True)

# DataFrame completo
imput_mean_b = mean_imput(dados_sim_b, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_median_b = median_imput(dados_sim_b, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_knn_b = knn_imput(dados_sim_b, metadados=metadados, variaveis=variaveis, return_reduced=True)